# LearnGuard — Dysgraphia Module: Dataset → Preprocessing → Features → Baseline

Follows the **Dysgraphia Detection Module** in the LearnGuard FYP-I mid-report (Sec. 3.2.1.2, 3.3.2, 3.3.5, 3.3.6):

| Report step | Where in this notebook |
|---|---|
| Resize, grayscale, histogram equalization, augmentation (rotation, brightness) | Sections 3 and 5 |
| Hand-crafted features (character-size variance, spacing consistency, stroke uniformity) | Section 4 |
| End-to-end CNN vs hand-crafted features | Sections 6 and 7 |
| SHAP explanation in plain language | Section 8 |
| Accuracy / Precision / Recall / F1 (+ sensitivity, specificity) | Sections 6 and 7 |

**Dataset:** *Potential Dysgraphia Handwriting Dataset of School-Age Children* (Mendeley Data, `39hr8dx76p`, v1).
249 handwriting images of 83 Malaysian children (ages 6–12): 114 *potential dysgraphia* (PD) and 135 *low potential dysgraphia* (LPD).

**Important design choices (do not remove):**
1. **Split by child, never by image.** 249 images come from only 83 children, so a random image split puts the same child in train and test and inflates scores.
2. **Augment only the training set, after the split.**
3. **No horizontal flips.** Mirrored letters are themselves a symptom, and flipping destroys slant information.
4. **Aspect-ratio-preserving resize.** Stretching a handwriting image distorts letter size and spacing, which are the very signals we measure.
5. Labels are *risk levels* ("potential" vs "low potential"), so this is a **screening aid, not a diagnosis**.

## 1. Setup

In [ ]:
import os, re, io, glob, zipfile, hashlib, warnings, random
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE      = "/content" if IN_COLAB else os.path.join(os.getcwd(), "work")
ZIP_PATH  = os.path.join(BASE, "dysgraphia.zip")
DATA_DIR  = os.path.join(BASE, "dysgraphia_raw")
OUT_DIR   = os.path.join(BASE, "preprocessed")
os.makedirs(BASE, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Target size (height, width). Handwriting lines are wide, so we use a wide canvas.
IMG_H, IMG_W = 128, 384
N_AUG_PER_IMAGE = 3          # augmented copies per TRAINING image
EPOCHS = int(os.environ.get("EPOCHS", 40))   # CNN max epochs (early stopping is on)
BACKUP_TO_DRIVE = False      # set True to copy outputs to Google Drive (recommended!)

print("Running in Colab:", IN_COLAB, "| working dir:", BASE)

## 2. Download the dataset from Mendeley
The cell tries Mendeley's public download endpoint. If your network blocks it, it falls back to a manual upload:
download the zip from https://data.mendeley.com/datasets/39hr8dx76p/1 in your browser, then upload it when prompted.

In [ ]:
import requests

def try_download(url, dest):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=120, stream=True)
        if r.status_code != 200:
            print(f"  HTTP {r.status_code} from {url}"); return False
        with open(dest, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        return zipfile.is_zipfile(dest)
    except Exception as e:
        print("  download failed:", e); return False

if os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH):
    print("Zip already present, skipping download.")
else:
    print("Downloading from Mendeley ...")
    ok = try_download("https://data.mendeley.com/public-api/zip/39hr8dx76p/download/1", ZIP_PATH)
    if not ok:
        print("Automatic download did not work. Please upload the zip manually.")
        if IN_COLAB:
            from google.colab import files
            up = files.upload()                       # choose the downloaded .zip
            name = next(iter(up))
            os.replace(name, ZIP_PATH)
        else:
            raise SystemExit(f"Place the dataset zip at {ZIP_PATH} and re-run.")

assert zipfile.is_zipfile(ZIP_PATH), "File is not a valid zip."
print("Zip OK:", os.path.getsize(ZIP_PATH) / 1e6, "MB")

In [ ]:
# Extract (handles zip-inside-zip, which Mendeley sometimes produces)
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

for _ in range(3):  # extract nested zips
    nested = [p for p in glob.glob(os.path.join(DATA_DIR, "**", "*.zip"), recursive=True)]
    if not nested: break
    for p in nested:
        with zipfile.ZipFile(p) as z:
            z.extractall(os.path.splitext(p)[0])
        os.remove(p)

for root, dirs, files_ in os.walk(DATA_DIR):
    if "__MACOSX" in root: continue
    depth = root[len(DATA_DIR):].count(os.sep)
    n_img = sum(f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")) for f in files_)
    print("  " * depth + f"{os.path.basename(root)}/  ({n_img} images)")

### 2.1 Build the file table and labels
Class folders are `potential dysgraphia` and `low potential dysgraphia`. Note that the second name *contains* the first,
so the code checks for "low" first. **Label 1 = potential dysgraphia (PD), label 0 = low potential dysgraphia (LPD).**

In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
rows = []
for root, _, fs in os.walk(DATA_DIR):
    if "__MACOSX" in root: continue
    for f in fs:
        if f.lower().endswith(EXTS) and not f.startswith("."):
            rows.append({"path": os.path.join(root, f), "folder": os.path.basename(root), "file": f})
df = pd.DataFrame(rows)
assert len(df) > 0, "No images found - check the extracted folder structure above."

def folder_to_label(name):
    n = name.lower()
    if "low" in n: return 0          # low potential dysgraphia
    if "potential" in n or "dysgraph" in n: return 1
    return -1

df["label"] = df["folder"].apply(folder_to_label)
print(df.groupby(["folder", "label"]).size())
assert set(df["label"]) == {0, 1}, "Could not map folders to classes - edit folder_to_label()."
print("\nTotal images:", len(df), "| PD:", (df.label == 1).sum(), "| LPD:", (df.label == 0).sum())
print("\nExample file names (needed to work out child IDs):")
print(df["file"].sample(min(12, len(df)), random_state=1).tolist())

### 2.2 Child IDs (critical for a leak-free split)
The dataset has 249 images from 83 children, so several images belong to one child. We must keep a child's images together.
The cell below guesses the child ID from the file name. **Look at the example file names printed above** and, if the guess is wrong,
set `CHILD_ID_REGEX` to a regex whose first group captures the child ID (e.g. `r"^(P\d+)"`).

In [ ]:
CHILD_ID_REGEX = None   # e.g. r"^(\d+)" or r"^([A-Za-z]+\d+)" -- set after inspecting file names

def guess_child_id(fname):
    stem = os.path.splitext(fname)[0].lower().strip()
    if CHILD_ID_REGEX:
        m = re.search(CHILD_ID_REGEX, os.path.splitext(fname)[0])
        return m.group(1) if m else stem
    stem = re.sub(r"\s*\(\d+\)$", "", stem)            # "name (2)"
    stem = re.sub(r"[\s_\-]+\d+$", "", stem)            # trailing "_2", "-3", " 4"
    return stem

df["child"] = df["file"].apply(guess_child_id)
n_children = df["child"].nunique()
print("Distinct child IDs found:", n_children, "(dataset description says 83)")

if n_children < 40 or n_children > len(df) * 0.95:
    print("\n⚠️  WARNING: the child-ID guess does not look right (expected ~83 groups).")
    print("   Falling back to one group per image. Test scores will then be OPTIMISTIC because")
    print("   the same child can appear in train and test. Set CHILD_ID_REGEX above and re-run.")
    df["child"] = df["path"]
    GROUPED_OK = False
else:
    GROUPED_OK = True
print("Grouping reliable:", GROUPED_OK)

## 3. Data audit

In [ ]:
def read_gray(path):
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)

bad, shapes, md5s = [], [], []
for p in df["path"]:
    img = read_gray(p)
    if img is None:
        bad.append(p); shapes.append((np.nan, np.nan)); md5s.append(None); continue
    shapes.append(img.shape)
    md5s.append(hashlib.md5(open(p, "rb").read()).hexdigest())
df["h"] = [s[0] for s in shapes]; df["w"] = [s[1] for s in shapes]; df["md5"] = md5s

print("Unreadable images:", len(bad))
df = df[~df["path"].isin(bad)].reset_index(drop=True)

dups = df[df.duplicated("md5", keep=False)]
print("Exact duplicate files:", len(dups))
if len(dups):
    # keep first copy only; flag label conflicts
    conflict = dups.groupby("md5")["label"].nunique().gt(1).sum()
    print("  duplicate groups with conflicting labels:", conflict)
    df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)

print("\nImage height/width summary:"); print(df[["h", "w"]].describe().round(1))
print("Median aspect ratio (w/h): %.2f   -> current canvas is %d x %d (ratio %.2f)" %
      ((df.w / df.h).median(), IMG_H, IMG_W, IMG_W / IMG_H))
print("Images per child (min/median/max):", df.groupby("child").size().agg(["min", "median", "max"]).tolist())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for r, (lab, title) in enumerate([(0, "Low potential dysgraphia (0)"), (1, "Potential dysgraphia (1)")]):
    sample = df[df.label == lab].sample(4, random_state=3)
    for c, p in enumerate(sample["path"]):
        axes[r][c].imshow(read_gray(p), cmap="gray"); axes[r][c].set_title(title, fontsize=9); axes[r][c].axis("off")
plt.suptitle("Raw samples", fontsize=14); plt.tight_layout(); plt.show()

## 4. Preprocessing (report Sec. 3.2.1.2)
Steps per image: grayscale → make background white → crop to the ink → aspect-ratio-preserving resize onto a fixed white canvas → CLAHE histogram equalization → scale to [0, 1].
The dataset images are already near-binary, so equalization changes little, but it is kept because the report specifies it and it helps if you later add photographed pages.

In [ ]:
def ensure_dark_ink_on_white(gray):
    return 255 - gray if gray.mean() < 127 else gray

def crop_to_ink(gray, pad=10):
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    pts = cv2.findNonZero(bw)
    if pts is None:
        return gray
    x, y, w, h = cv2.boundingRect(pts)
    H, W = gray.shape
    return gray[max(0, y - pad):min(H, y + h + pad), max(0, x - pad):min(W, x + w + pad)]

def resize_keep_aspect(gray, out_h=IMG_H, out_w=IMG_W, fill=255):
    h, w = gray.shape
    scale = min(out_h / h, out_w / w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    resized = cv2.resize(gray, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.full((out_h, out_w), fill, np.uint8)
    y0, x0 = (out_h - nh) // 2, (out_w - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = resized
    return canvas

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess(path, return_cropped=False):
    gray = read_gray(path)
    gray = ensure_dark_ink_on_white(gray)
    cropped = crop_to_ink(gray)
    canvas = resize_keep_aspect(cropped)
    eq = _clahe.apply(canvas)
    out = eq.astype(np.float32) / 255.0
    return (out, cropped) if return_cropped else out

X = np.stack([preprocess(p) for p in df["path"]])[..., None]    # (N, H, W, 1)
y = df["label"].values.astype(int)
groups = df["child"].values
print("X:", X.shape, "| pixel range:", float(X.min()), "-", float(X.max()), "| PD:", int(y.sum()), "LPD:", int((1 - y).sum()))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 5))
for i, idx in enumerate(df.sample(4, random_state=5).index):
    axes[0][i].imshow(read_gray(df.path[idx]), cmap="gray"); axes[0][i].set_title("before"); axes[0][i].axis("off")
    axes[1][i].imshow(X[idx, ..., 0], cmap="gray"); axes[1][i].set_title(f"after ({IMG_H}x{IMG_W}), label={y[idx]}"); axes[1][i].axis("off")
plt.tight_layout(); plt.show()

## 5. Child-level train/test split, then augmentation (train only)
`StratifiedGroupKFold` keeps every child's images in a single fold while balancing the two classes. Fold 0 becomes the test set (~20%).
A second grouped split carves a validation set out of the training children.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def grouped_split(idx, y, groups, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    tr, te = next(sgkf.split(idx, y[idx], groups[idx]))
    return idx[tr], idx[te]

all_idx = np.arange(len(df))
trainval_idx, test_idx = grouped_split(all_idx, y, groups, 5, SEED)
train_idx, val_idx     = grouped_split(trainval_idx, y, groups, 5, SEED + 1)

def describe(name, idx):
    print(f"{name:6s}: {len(idx):4d} images | {len(set(groups[idx])):3d} children | PD={int(y[idx].sum()):3d} LPD={int((1-y[idx]).sum()):3d}")
for n, i in [("train", train_idx), ("val", val_idx), ("test", test_idx)]: describe(n, i)

# Leakage check: no child in two sets
assert not (set(groups[train_idx]) & set(groups[test_idx])), "child leakage train/test"
assert not (set(groups[train_idx]) & set(groups[val_idx])),  "child leakage train/val"
assert not (set(groups[val_idx]) & set(groups[test_idx])),   "child leakage val/test"
print("✅ No child appears in more than one split." if GROUPED_OK else "⚠️ Split is per-image (child IDs unknown) - see warning above.")

In [ ]:
rng = np.random.default_rng(SEED)

def augment(img):
    # img: float32 (H, W) in [0, 1], white background. Rotation, shift, scale, brightness. NO flips.
    h, w = img.shape
    angle = rng.uniform(-7, 7)
    scale = rng.uniform(0.92, 1.05)
    tx, ty = rng.uniform(-0.03, 0.03) * w, rng.uniform(-0.05, 0.05) * h
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale); M[:, 2] += (tx, ty)
    out = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=1.0)
    out = out * rng.uniform(0.85, 1.10) + rng.uniform(-0.05, 0.05)       # brightness / contrast jitter
    return np.clip(out, 0, 1).astype(np.float32)

X_train_orig, y_train_orig = X[train_idx], y[train_idx]
aug_X, aug_y = [X_train_orig], [y_train_orig]
for _ in range(N_AUG_PER_IMAGE):
    aug_X.append(np.stack([augment(im[..., 0]) for im in X_train_orig])[..., None])
    aug_y.append(y_train_orig)
X_train = np.concatenate(aug_X); y_train = np.concatenate(aug_y)
X_val, y_val   = X[val_idx],  y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("Train (after augmentation):", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
axes[0].imshow(X_train_orig[0, ..., 0], cmap="gray"); axes[0].set_title("original")
for i in range(1, 5):
    axes[i].imshow(augment(X_train_orig[0, ..., 0]), cmap="gray"); axes[i].set_title(f"augmented {i}")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Save preprocessed arrays (test/val are NOT augmented)
np.savez_compressed(os.path.join(OUT_DIR, "dysgraphia_cnn_data.npz"),
                    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test)
df.drop(columns=["md5"]).assign(split=lambda d: np.select(
    [d.index.isin(train_idx), d.index.isin(val_idx), d.index.isin(test_idx)], ["train", "val", "test"], "none")
).to_csv(os.path.join(OUT_DIR, "file_table_with_splits.csv"), index=False)
print("Saved to", OUT_DIR, os.listdir(OUT_DIR))

if BACKUP_TO_DRIVE and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    import shutil; shutil.copytree(OUT_DIR, "/content/drive/MyDrive/learnguard_dysgraphia_preprocessed", dirs_exist_ok=True)
    print("Backed up to Google Drive.")

## 6. Hand-crafted features (report Sec. 3.3.2)
Features capture what a teacher can understand: **character size variance, inter-letter spacing consistency, stroke uniformity, ink density, baseline wobble**.
They are computed on the cropped (not resized) binary image so that sizes are comparable across children.

In [ ]:
from skimage.morphology import skeletonize

FEATURE_NAMES = [
    "ink_density", "num_components", "char_height_cv", "char_width_cv", "char_size_ratio_cv",
    "letter_gap_cv", "mean_letter_gap_rel", "baseline_wobble", "stroke_width_mean_rel", "stroke_width_cv",
    "touching_letters_ratio",
]

def handcrafted_features(path):
    gray = ensure_dark_ink_on_white(read_gray(path))
    gray = crop_to_ink(gray, pad=2)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    bw = (bw > 0).astype(np.uint8)
    H, W = bw.shape
    ink = bw.sum()
    f = dict.fromkeys(FEATURE_NAMES, 0.0)
    f["ink_density"] = ink / (H * W)

    n, lab, stats, cent = cv2.connectedComponentsWithStats(bw, connectivity=8)
    comps = [(stats[i, 0], stats[i, 1], stats[i, 2], stats[i, 3], stats[i, 4]) for i in range(1, n)]
    min_area = max(15, 0.0003 * H * W)
    comps = [c for c in comps if c[4] >= min_area]
    f["num_components"] = len(comps)

    cv_ = lambda a: float(np.std(a) / (np.mean(a) + 1e-6))
    if len(comps) >= 3:
        xs, ys, ws, hs, _ = map(np.array, zip(*comps))
        f["char_height_cv"] = cv_(hs); f["char_width_cv"] = cv_(ws); f["char_size_ratio_cv"] = cv_(ws / (hs + 1e-6))
        order = np.argsort(xs); xs, ws, ys, hs = xs[order], ws[order], ys[order], hs[order]
        gaps = xs[1:] - (xs[:-1] + ws[:-1])                          # horizontal gap between neighbours
        pos_gaps = gaps[gaps > 0]
        if len(pos_gaps) >= 2:
            f["letter_gap_cv"] = cv_(pos_gaps); f["mean_letter_gap_rel"] = float(pos_gaps.mean() / (hs.mean() + 1e-6))
        f["touching_letters_ratio"] = float((gaps <= 0).mean())
        f["baseline_wobble"] = float(np.std(ys + hs) / (np.mean(hs) + 1e-6))

    skel = skeletonize(bw > 0)
    if skel.sum() > 0:
        dist = cv2.distanceTransform(bw, cv2.DIST_L2, 3)
        sw = 2 * dist[skel]
        f["stroke_width_mean_rel"] = float(sw.mean() / (H + 1e-6)); f["stroke_width_cv"] = cv_(sw)
    return f

feat_df = pd.DataFrame([handcrafted_features(p) for p in df["path"]])[FEATURE_NAMES]
feat_df = feat_df.replace([np.inf, -np.inf], 0).fillna(0)
print(feat_df.describe().T[["mean", "std", "min", "max"]].round(3))
feat_df.assign(label=y, child=groups).to_csv(os.path.join(OUT_DIR, "handcrafted_features.csv"), index=False)

print("\nMean feature value by class (1 = potential dysgraphia):")
print(feat_df.assign(label=y).groupby("label").mean().T.round(3))

## 7. Baselines (sanity check, evaluated on unseen children)
**Baseline A:** Random Forest on hand-crafted features, 5-fold *grouped* CV. **Baseline B:** small CNN on the preprocessed images.
With ~83 children expect modest, noisy numbers. Treat these as a pipeline check, not a final result. The report's 90% target is not reliable to claim on a dataset this small.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

def report(y_true, y_pred, y_prob=None, title=""):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"--- {title} ---")
    print(f"Accuracy   {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision  {precision_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"Recall/Sensitivity {recall_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"Specificity {tn / (tn + fp + 1e-9):.3f}")
    print(f"F1         {f1_score(y_true, y_pred, zero_division=0):.3f}")
    if y_prob is not None and len(set(y_true)) == 2: print(f"ROC-AUC    {roc_auc_score(y_true, y_prob):.3f}")
    print(f"Confusion matrix [[TN FP],[FN TP]]: [[{tn} {fp}],[{fn} {tp}]]\n")

rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced", random_state=SEED, n_jobs=-1)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
prob = cross_val_predict(rf, feat_df.values, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
report(y, (prob >= 0.5).astype(int), prob, "Random Forest on hand-crafted features (grouped 5-fold CV)")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
tf.random.set_seed(SEED)

def build_cnn(input_shape=(IMG_H, IMG_W, 1)):
    m = models.Sequential([
        layers.Input(input_shape),
        layers.Conv2D(16, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5), layers.Dense(32, activation="relu"), layers.Dropout(0.3), layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m

cnn = build_cnn()
cw = {0: len(y_train) / (2 * (y_train == 0).sum()), 1: len(y_train) / (2 * (y_train == 1).sum())}
hist = cnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=16, class_weight=cw, verbose=2,
               callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)])

p_test = cnn.predict(X_test, verbose=0).ravel()
report(y_test, (p_test >= 0.5).astype(int), p_test, "Small CNN on preprocessed images (held-out children)")
describe("test", test_idx)

plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1); plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val"); plt.legend(); plt.title("loss")
plt.subplot(1, 2, 2); plt.plot(hist.history["accuracy"], label="train"); plt.plot(hist.history["val_accuracy"], label="val"); plt.legend(); plt.title("accuracy")
plt.tight_layout(); plt.show()

## 8. Explainability (report Sec. 3.3.5): SHAP → plain-language sentence
We fit the Random Forest on the training children and explain predictions for the test children. The helper turns the top SHAP features into
a sentence a teacher can read.

In [ ]:
import shap

tr_rows, te_rows = np.concatenate([train_idx, val_idx]), test_idx
rf_final = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced", random_state=SEED, n_jobs=-1)
rf_final.fit(feat_df.values[tr_rows], y[tr_rows])
report(y[te_rows], rf_final.predict(feat_df.values[te_rows]), rf_final.predict_proba(feat_df.values[te_rows])[:, 1],
       "Random Forest (held-out children)")

expl = shap.TreeExplainer(rf_final)
sv = expl.shap_values(feat_df.values[te_rows])
sv_pos = sv[1] if isinstance(sv, list) else (sv[..., 1] if sv.ndim == 3 else sv)    # contributions toward class 1

shap.summary_plot(sv_pos, feat_df.iloc[te_rows], show=True)

PLAIN = {
    "ink_density": "overall amount of ink on the page", "num_components": "number of separate letter/word pieces",
    "char_height_cv": "uneven letter heights", "char_width_cv": "uneven letter widths", "char_size_ratio_cv": "inconsistent letter shapes",
    "letter_gap_cv": "irregular spacing between letters", "mean_letter_gap_rel": "spacing between letters",
    "baseline_wobble": "writing that drifts above/below the line", "stroke_width_mean_rel": "pen pressure / line thickness",
    "stroke_width_cv": "uneven line thickness", "touching_letters_ratio": "letters touching or overlapping",
}

def explain_student(local_i, top_k=3):
    prob = rf_final.predict_proba(feat_df.values[te_rows][local_i:local_i + 1])[0, 1]
    contrib = sv_pos[local_i]
    order = np.argsort(-np.abs(contrib))[:top_k]
    parts = [f"{PLAIN[FEATURE_NAMES[j]]} ({'raises' if contrib[j] > 0 else 'lowers'} the risk)" for j in order]
    level = "HIGH" if prob >= 0.66 else "MODERATE" if prob >= 0.4 else "LOW"
    return (f"Dysgraphia risk: {level} ({prob:.0%}). Main signals in this handwriting: " + "; ".join(parts) +
            ". This is a screening result, not a diagnosis; consider referral to a specialist if risk is high.")

for i in range(min(3, len(te_rows))):
    print(f"\nTest image {i} (true label: {'potential' if y[te_rows][i] else 'low potential'} dysgraphia)")
    print(explain_student(i))

## 9. Export the model for the backend / frontend
This section saves everything the backend needs into **Google Drive** (`MyDrive/learnguard_artifacts/dysgraphia/`):

| File | What it is |
|---|---|
| `dysgraphia_rf.joblib` | Random Forest trained on **all** children (for deployment) |
| `model_config.json` | Feature names, plain-language names, risk thresholds, library versions, disclaimers |
| `parity_samples.json` | A few images with the exact probabilities from this notebook, so the backend can prove it gives the same answers |
| `dysgraphia_cnn.keras` | (optional) the CNN, not needed by the backend |

The numbers you **report** come from the child-level split in Sections 6-7. The exported model is refit on all children, as is normal for deployment, so do not quote its accuracy.

In [ ]:
import joblib, json, datetime, sys, sklearn, skimage

rf_deploy = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced",
                                   random_state=SEED, n_jobs=-1).fit(feat_df.values, y)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ART_DIR = "/content/drive/MyDrive/learnguard_artifacts/dysgraphia"
else:
    ART_DIR = os.path.join(OUT_DIR, "artifacts")
os.makedirs(ART_DIR, exist_ok=True)

joblib.dump(rf_deploy, os.path.join(ART_DIR, "dysgraphia_rf.joblib"), compress=3)

config = {
    "module": "dysgraphia",
    "model_version": "dysgraphia-rf-" + datetime.date.today().isoformat(),
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "feature_names": FEATURE_NAMES,
    "plain_names": PLAIN,
    "risk_thresholds": {"moderate": 0.40, "high": 0.66},
    "positive_class": "potential dysgraphia (label 1)",
    "trained_on": {"dataset": "Mendeley 39hr8dx76p (Malay handwriting, 83 children)", "n_images": int(len(df)),
                   "n_children": int(df["child"].nunique()), "child_ids_reliable": bool(GROUPED_OK)},
    "versions": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "numpy": np.__version__,
                 "opencv": cv2.__version__, "scikit-image": skimage.__version__},
    "warnings": ["Screening aid, not a diagnosis.",
                 "Trained on Malay-language handwriting from 83 children; validate on English handwriting before relying on it."],
}
json.dump(config, open(os.path.join(ART_DIR, "model_config.json"), "w"), indent=2)

# parity samples: the backend must reproduce these probabilities from the raw images
samp = df.sample(min(6, len(df)), random_state=7)
parity = []
for i in samp.index:
    parity.append({"file": df.file[i], "folder": df.folder[i], "label": int(y[i]),
                   "expected_probability": float(rf_deploy.predict_proba(feat_df.values[[i]])[0, 1]),
                   "features": {n: float(v) for n, v in zip(FEATURE_NAMES, feat_df.values[i])}})
json.dump(parity, open(os.path.join(ART_DIR, "parity_samples.json"), "w"), indent=2)

try:
    cnn.save(os.path.join(ART_DIR, "dysgraphia_cnn.keras"))
except Exception as e:
    print("CNN not saved (optional):", repr(e)[:120])

# verify the saved model reloads and predicts identically
re_loaded = joblib.load(os.path.join(ART_DIR, "dysgraphia_rf.joblib"))
assert np.allclose(re_loaded.predict_proba(feat_df.values), rf_deploy.predict_proba(feat_df.values)), "reload mismatch"
print("Saved to:", ART_DIR)
for f in sorted(os.listdir(ART_DIR)): print(f"  {f:28s} {os.path.getsize(os.path.join(ART_DIR, f)) / 1e6:7.2f} MB")
print("\nParity images (keep these files handy for the backend test):", [p["file"] for p in parity])

## 10. What to report honestly in your FYP
- Dataset: 249 images, 83 children, **risk-level labels** (not clinical diagnoses), Malay sentences, scanned paper. Say that results may not transfer to English or Urdu handwriting.
- Evaluation: child-level grouped split; report **sensitivity and specificity**, not only accuracy. With this sample size the confidence intervals are wide.
- The report's NFR of ≥90% accuracy is unlikely to be demonstrated reliably on this dataset alone. Consider rewording it as a target to be validated on larger, locally collected data (with consent and ethics approval).
- Next steps from the report: try a lightweight **ResNet** (transfer learning), a **CNN + LSTM** only if you obtain stroke-order data, and collect your own classroom handwriting (Pakistan / English / Urdu) for fine-tuning.